# Zero-Inflated and Two-Part Mixed Effects Models (Python)

**Author:** Dimitris Rizopoulos

This notebook accompanies the R vignette *Zero-Inflated and Two-Part Mixed
Effects Models* and documents the planned Python API in **glmmadaptive**.

> **Current status:** Zero-inflated and hurdle families are **not yet
> implemented** in the Python port.  The classes exist as stubs that expose
> the correct attributes but raise `NotImplementedError` for all
> computational methods.  Full implementations will be available in a future
> release.

| Family (R) | Python class | Status |
|---|---|---|
| `zi.poisson()` | `ZIPoisson` | Stub — not yet implemented |
| `zi.negative.binomial()` | `ZINegativeBinomial` | Stub — not yet implemented |
| `zi.binomial()` | `ZIBinomial` | Stub — not yet implemented |
| `hurdle.lognormal()` | `HurdleLogNormal` | Stub — not yet implemented |
| `hurdle.poisson()` | `HurdlePoisson` | Stub — not yet implemented |
| `hurdle.negative.binomial()` | `HurdleNegativeBinomial` | Stub — not yet implemented |

In [ ]:
import warnings
warnings.filterwarnings("ignore")

from glmmadaptive.families import (
    ZIPoisson, ZINegativeBinomial,
    HurdlePoisson, HurdleNegativeBinomial, HurdleLogNormal,
)

# All stubs declare the correct metadata:
for cls in [ZIPoisson, ZINegativeBinomial, HurdlePoisson,
            HurdleNegativeBinomial, HurdleLogNormal]:
    fam = cls()
    print(f"{cls.__name__:30s}  has_zi={fam.has_zi}  link={fam.link}  n_phis={fam.n_phis}")

---

## 1  Zero-Inflated Poisson Mixed Effects Model

The ZIP model uses a logistic regression to model structural zeros and a
Poisson distribution for the non-zero counts.

**R code (reference implementation):**

```r
fm1 <- mixed_model(
    y ~ sex * time,
    random   = ~ 1 | id,
    data     = DF,
    family   = zi.poisson(),
    zi_fixed = ~ sex
)

# Extend with a random intercept in the zero part
fm2 <- update(fm1, zi_random = ~ 1 | id)
anova(fm1, fm2)
```

**Planned Python API:**

```python
# Raises NotImplementedError in current version
from glmmadaptive import MixedModel
from glmmadaptive.families import ZIPoisson

fm1 = MixedModel(
    fixed    = "y ~ sex * time",
    random   = "~ 1 | id",
    data     = DF,
    family   = ZIPoisson(),
    zi_fixed = "~ sex",
).fit()

fm2 = MixedModel(
    fixed     = "y ~ sex * time",
    random    = "~ 1 | id",
    data      = DF,
    family    = ZIPoisson(),
    zi_fixed  = "~ sex",
    zi_random = "~ 1 | id",
).fit()
```

---

## 2  Zero-Inflated Negative Binomial Mixed Effects Model

The ZINB model extends ZIP by adding an over-dispersion parameter
$\theta = \exp(\phi)$ to the non-zero part.

**R code:**

```r
gm1 <- mixed_model(
    y ~ sex * time,
    random   = ~ 1 | id,
    data     = DF,
    family   = zi.negative.binomial(),
    zi_fixed = ~ sex
)

# Non-nested comparison (no p-value)
anova(gm1, fm2, test = FALSE)
```

**Planned Python API:**

```python
# Raises NotImplementedError in current version
from glmmadaptive.families import ZINegativeBinomial

gm1 = MixedModel(
    fixed    = "y ~ sex * time",
    random   = "~ 1 | id",
    data     = DF,
    family   = ZINegativeBinomial(),
    zi_fixed = "~ sex",
).fit()
```

---

## 3  Two-Part Mixed Effects Model for Semi-Continuous Data (HurdleLogNormal)

Models continuous data with excess zeros: logistic regression for the
zero/non-zero split and a log-normal mixed model for the positive part.
The dispersion parameter `exp(phis)` gives the standard deviation of the
log-normal errors.

**R code:**

```r
km1 <- mixed_model(
    y ~ sex * time, random = ~ 1 | id, data = DF,
    family = hurdle.lognormal(), n_phis = 1, zi_fixed = ~ sex
)
km2 <- update(km1, random = ~ 1 || id, zi_random = ~ 1 | id)
marginal_coefs(km2)
```

**Planned Python API:**

```python
# Raises NotImplementedError in current version
from glmmadaptive.families import HurdleLogNormal

km1 = MixedModel(
    fixed    = "y ~ sex * time",
    random   = "~ 1 | id",
    data     = DF,
    family   = HurdleLogNormal(),
    zi_fixed = "~ sex",
).fit()
```

---

## 4  Two-Part / Hurdle Poisson Mixed Effects Model

Uses a logistic regression for the zero/non-zero split and a zero-truncated
Poisson for the positive counts.  Fixed-effects coefficients relate to the
mean $\mu$ of the **full** (untruncated) Poisson, not to the conditional
mean $\mu / (1 - e^{-\mu})$ among positive counts.

**R code:**

```r
dm1 <- mixed_model(
    y ~ sex * time, random = ~ time | id, data = DF,
    family = hurdle.poisson(), zi_fixed = ~ sex
)
dm2 <- update(dm1, zi_random = ~ 1 | id)
anova(dm1, dm2)
```

**Planned Python API:**

```python
# Raises NotImplementedError in current version
from glmmadaptive.families import HurdlePoisson

dm1 = MixedModel(
    fixed    = "y ~ sex * time",
    random   = "~ time | id",
    data     = DF,
    family   = HurdlePoisson(),
    zi_fixed = "~ sex",
).fit()
```

---

## 5  Two-Part / Hurdle Negative Binomial Mixed Effects Model

Identical in structure to the hurdle Poisson family, but replaces the
zero-truncated Poisson with a zero-truncated negative binomial distribution
to accommodate over-dispersion in the positive counts.

**R code:**

```r
hm1 <- mixed_model(
    y ~ sex * time, random = ~ time | id, data = DF,
    family = hurdle.negative.binomial(), zi_fixed = ~ sex
)
hm2 <- update(hm1, zi_random = ~ 1 | id)
anova(hm1, hm2)
```

**Planned Python API:**

```python
# Raises NotImplementedError in current version
from glmmadaptive.families import HurdleNegativeBinomial

hm1 = MixedModel(
    fixed    = "y ~ sex * time",
    random   = "~ time | id",
    data     = DF,
    family   = HurdleNegativeBinomial(),
    zi_fixed = "~ sex",
).fit()
```